# Llama3 Fine-Tuning for Diabetes Recommendations
**Using Unsloth LoRA on Google Colab (T4 GPU — Free Tier)**

### CRITICAL: Instructions Following a Crash or Restart
If your session crashed or you clicked **Runtime -> Restart Session**:
1. You **MUST** run **Step 1** (Installs) again (it will be instant if already installed).
2. You **MUST** run **Step 3**, **Step 4**, and **Step 5** to reload the model and data into memory.
3. Then you can continue with **Step 6** (Training).

---

## Step 1 — Install Unsloth & Dependencies

In [ ]:
# Update transformers and install Unsloth & dependencies from GitHub
!pip install --upgrade transformers -q
!pip install "unsloth_zoo @ git+https://github.com/unslothai/unsloth-zoo.git" -q
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q
!pip install --no-deps trl peft accelerate bitsandbytes -q

import unsloth # Import early to enable patches
print('Installation and initial patching complete.')

## Step 2 — Mount Google Drive & Load Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import json
import os

DRIVE_DIR = '/content/drive/MyDrive/diabetes_finetune'
TRAIN_PATH = f'{DRIVE_DIR}/train.jsonl'
VAL_PATH   = f'{DRIVE_DIR}/val.jsonl'

def load_jsonl(path):
    records = []
    if not os.path.exists(path):
        print(f'Error: File not found at {path}. Please check your Google Drive path.')
        return records
    
    with open(path, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                try: records.append(json.loads(line))
                except: continue
    return records

train_data = load_jsonl(TRAIN_PATH)
val_data   = load_jsonl(VAL_PATH)

print(f'Records loaded: {len(train_data)} train, {len(val_data)} val')

## Step 3 — Load Llama3-8B-Instruct (4-bit)

In [ ]:
from unsloth import FastLanguageModel
import torch
import gc

# Clear memory
gc.collect()
torch.cuda.empty_cache()

MAX_SEQ_LENGTH = 1024 
model_name = "unsloth/llama-3-8b-instruct-bnb-4bit"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

print(f'Model {model_name} loaded successfully.')

## Step 4 — Attach LoRA Adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=32,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
print('LoRA adapters attached.')

## Step 5 — Prepare Dataset

In [ ]:
from datasets import Dataset

if 'tokenizer' not in globals():
    raise NameError("Please run Step 3 before this cell.")

def format_to_chatml(record):
    return tokenizer.apply_chat_template(
        record['messages'],
        tokenize=False,
        add_generation_prompt=False
    )

train_dataset = Dataset.from_dict({'text': [format_to_chatml(r) for r in train_data]})
val_dataset   = Dataset.from_dict({'text': [format_to_chatml(r) for r in val_data]})
print(f'Datasets prepared. Train size: {len(train_dataset)}')

## Step 6 — Configure and Run Training

In [ ]:
import unsloth # Ensure unsloth is imported first internally
from trl import SFTTrainer
from transformers import TrainingArguments
import os

# Safety check for session restarts
if 'model' not in globals() or 'train_dataset' not in globals():
    raise NameError("Environment lost. You must re-run Step 3, 4, and 5 before starting training.")

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

training_args = TrainingArguments(
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    warmup_steps=10,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=not unsloth.is_bfloat16_supported(),
    bf16=unsloth.is_bfloat16_supported(),
    logging_steps=5,
    optim='adamw_8bit',
    output_dir=f'{DRIVE_DIR}/llama3_diabetes_adapter',
    save_strategy='epoch',
    eval_strategy='epoch',
    load_best_model_at_end=True,
    weight_decay=0.01,
    lr_scheduler_type='linear',
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    dataset_text_field='text',
    max_seq_length=1024,
    args=training_args,
)

print('Starting training...')
trainer.train()

## Step 7 — Save LoRA Adapter

In [ ]:
model.save_pretrained(f'{DRIVE_DIR}/llama3_diabetes_adapter')
tokenizer.save_pretrained(f'{DRIVE_DIR}/llama3_diabetes_adapter')
print('Adapter saved.')

## Step 8 — Export to GGUF for Ollama

In [ ]:
model.save_pretrained_gguf(
    f'{DRIVE_DIR}/gguf_export',
    tokenizer,
    quantization_method = 'q4_k_m',
)